In [ ]:
%pip install -q "pandas==2.2.3" "bottleneck>=1.5.0" "ultralytics>=8.3.0,<9"


In [ ]:
import csv
import shutil
import subprocess
from collections import Counter, deque
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image
from IPython.display import HTML, display
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("GPU available:", torch.cuda.is_available())
print("Selected device:", DEVICE)
print("OpenCV:", cv2.__version__)
print("Torch:", torch.__version__)


In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No video was uploaded. Run this cell again and choose an MP4 file.")

video_path = str(Path(next(iter(uploaded))).resolve())
print("Uploaded video:", video_path)
print("File size (MB):", round(Path(video_path).stat().st_size / (1024 * 1024), 2))


In [ ]:
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise RuntimeError("Cannot open the video. Try an MP4/H.264 file.")

fps = float(cap.get(cv2.CAP_PROP_FPS))
if fps <= 0 or fps > 240:
    print("FPS metadata missing or unusual — defaulting to 25.")
    fps = 25.0

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
ok, first_frame = cap.read()
cap.release()
if not ok or first_frame is None:
    raise RuntimeError("Video opened but no readable frames found. Try another MP4.")

source_height, source_width = first_frame.shape[:2]
print("Resolution:", source_width, "x", source_height)
print("FPS:", round(fps, 2))
print("Frames (metadata):", frame_count)
if frame_count > 0:
    print("Approx. duration (s):", round(frame_count / fps, 2))

preview_w = min(900, source_width)
preview_h = max(1, round(source_height * preview_w / source_width))
preview = cv2.resize(first_frame, (preview_w, preview_h))
display(Image.fromarray(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB)))


In [ ]:
MODEL_NAME = "yolov8n.pt"

TRACKER = "bytetrack.yaml"

VEHICLE_CLASSES = [2, 3, 5, 7]
CLASS_NAMES     = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
CLASS_COLORS    = {2: (0, 200, 255), 3: (0, 165, 255),
                   5: (50, 205, 50), 7: (147, 20, 255)}

CONFIDENCE = 0.20

IOU_THRESHOLD = 0.70

if DEVICE != "cpu":
    RESIZE_WIDTH = 1280
    IMAGE_SIZE   = 1280
else:
    RESIZE_WIDTH = 960
    IMAGE_SIZE   = 960

LINE_DIRECTION = "horizontal"

LINE_RATIO = 0.75

ROAD_LEFT_RATIO  = 0.08
ROAD_RIGHT_RATIO = 0.95

ROAD_TOP_RATIO    = 0.35
ROAD_BOTTOM_RATIO = 0.95

ROI_RATIOS = [
    (0.05, 0.10),
    (0.95, 0.10),
    (0.95, 0.98),
    (0.05, 0.98),
]

HYSTERESIS_MARGIN = 3

TRAJECTORY_LEN = 6

MIN_TRACK_FRAMES = 2

DEDUP_RADIUS = 0
DEDUP_FRAMES = 0
DEDUP_CLASS_RULES = {
    "car": (0, 0),
    "motorcycle": (0, 0),
    "bus": (45, 15),
    "truck": (45, 15),
}
ENABLE_SPATIAL_DEDUP = True

MAX_SECONDS = None

assert LINE_DIRECTION in ("horizontal", "vertical"), "LINE_DIRECTION must be 'horizontal' or 'vertical'"
assert 0.1 <= LINE_RATIO <= 0.9, "LINE_RATIO must be between 0.1 and 0.9"
print("Configuration OK")
print(f"  Model   : {MODEL_NAME}")
print(f"  Tracker : {TRACKER}")
print(f"  conf    : {CONFIDENCE}  iou: {IOU_THRESHOLD}  imgsz: {IMAGE_SIZE}")
print(f"  Line    : {LINE_DIRECTION} @ {LINE_RATIO:.0%}")
print(f"  Hyster  : {HYSTERESIS_MARGIN}px  Traj: {TRAJECTORY_LEN}  MinFrames: {MIN_TRACK_FRAMES}")
print(f"  Dedup   : enabled={ENABLE_SPATIAL_DEDUP}, class rules={DEDUP_CLASS_RULES}")
print(f"  ROI     : {'enabled' if ROI_RATIOS else 'disabled'}")


In [ ]:
W = min(RESIZE_WIDTH, source_width)
W += W % 2
H = max(2, int(round(source_height * W / source_width)))
H += H % 2
print(f"Working resolution: {W} x {H}")

if LINE_DIRECTION == "horizontal":
    line_coordinate = int(H * LINE_RATIO)
    line_start = (int(W * ROAD_LEFT_RATIO),  line_coordinate)
    line_end   = (int(W * ROAD_RIGHT_RATIO), line_coordinate)
else:
    line_coordinate = int(W * LINE_RATIO)
    line_start = (line_coordinate, int(H * ROAD_TOP_RATIO))
    line_end   = (line_coordinate, int(H * ROAD_BOTTOM_RATIO))

print(f"Counting line: {line_start} -> {line_end}  (coordinate={line_coordinate})")

roi_polygon = None
if ROI_RATIOS is not None:
    roi_pts = np.array(
        [(int(rx * W), int(ry * H)) for rx, ry in ROI_RATIOS], dtype=np.int32
    ).reshape((-1, 1, 2))
    roi_polygon = roi_pts
    print("ROI polygon (px):", roi_pts.reshape(-1, 2).tolist())
else:
    print("ROI: disabled")

preview_frame = cv2.resize(first_frame, (W, H))
preview_vis   = preview_frame.copy()

if roi_polygon is not None:
    overlay = preview_vis.copy()
    cv2.fillPoly(overlay, [roi_polygon], (0, 255, 0))
    cv2.addWeighted(overlay, 0.15, preview_vis, 0.85, 0, preview_vis)
    cv2.polylines(preview_vis, [roi_polygon], isClosed=True, color=(0, 220, 0), thickness=2)

cv2.line(preview_vis, line_start, line_end, (0, 0, 255), 3)

if LINE_DIRECTION == "horizontal":
    cv2.line(preview_vis, (line_start[0], line_coordinate - HYSTERESIS_MARGIN),
             (line_end[0],   line_coordinate - HYSTERESIS_MARGIN), (0, 100, 200), 1)
    cv2.line(preview_vis, (line_start[0], line_coordinate + HYSTERESIS_MARGIN),
             (line_end[0],   line_coordinate + HYSTERESIS_MARGIN), (0, 100, 200), 1)
else:
    cv2.line(preview_vis, (line_coordinate - HYSTERESIS_MARGIN, line_start[1]),
             (line_coordinate - HYSTERESIS_MARGIN, line_end[1]),   (0, 100, 200), 1)
    cv2.line(preview_vis, (line_coordinate + HYSTERESIS_MARGIN, line_start[1]),
             (line_coordinate + HYSTERESIS_MARGIN, line_end[1]),   (0, 100, 200), 1)

cv2.circle(preview_vis, line_start, 7, (0, 255, 255), -1)
cv2.circle(preview_vis, line_end,   7, (0, 255, 255), -1)

cv2.putText(preview_vis, f"Counting line ({LINE_DIRECTION})",
            (line_start[0] + 5, line_start[1] - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2)
if roi_polygon is not None:
    cv2.putText(preview_vis, "ROI (green)",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 220, 0), 2)

display(Image.fromarray(cv2.cvtColor(preview_vis, cv2.COLOR_BGR2RGB)))


In [ ]:
preview_model = YOLO(MODEL_NAME)
det = preview_model.predict(
    source=preview_frame,
    classes=VEHICLE_CLASSES,
    conf=CONFIDENCE,
    iou=IOU_THRESHOLD,
    imgsz=IMAGE_SIZE,
    device=DEVICE,
    verbose=False,
)[0]

annotated = det.plot()

if roi_polygon is not None:
    overlay2 = annotated.copy()
    cv2.fillPoly(overlay2, [roi_polygon], (0, 255, 0))
    cv2.addWeighted(overlay2, 0.15, annotated, 0.85, 0, annotated)
    cv2.polylines(annotated, [roi_polygon], True, (0, 220, 0), 2)
cv2.line(annotated, line_start, line_end, (0, 0, 255), 3)

print(f"Detections in first frame: {len(det.boxes)}")
for b in det.boxes:
    cls_id = int(b.cls)
    print(f"  {CLASS_NAMES.get(cls_id, cls_id)}  conf={float(b.conf):.3f}")

display(Image.fromarray(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)))


In [ ]:
def inside_roi(cx, cy, polygon):
    if polygon is None:
        return True
    return cv2.pointPolygonTest(polygon, (float(cx), float(cy)), False) >= 0

def stable_side(point, line_coord, line_direction, margin):
    """-1 or +1 only after a centre is OUTSIDE the hysteresis band."""
    coord = point[1] if line_direction == "horizontal" else point[0]
    if coord < line_coord - margin:
        return -1
    if coord > line_coord + margin:
        return +1
    return 0

def crossing_point(a, b, start, end, line_direction):
    """Return the intersection with the finite counting segment, or None."""
    if line_direction == "horizontal":
        dy = b[1] - a[1]
        if dy == 0:
            return None
        t = (start[1] - a[1]) / dy
        coord = a[0] + t * (b[0] - a[0])
        if 0 <= t <= 1 and min(start[0], end[0]) <= coord <= max(start[0], end[0]):
            return (float(coord), float(start[1]))
    else:
        dx = b[0] - a[0]
        if dx == 0:
            return None
        t = (start[0] - a[0]) / dx
        coord = a[1] + t * (b[1] - a[1])
        if 0 <= t <= 1 and min(start[1], end[1]) <= coord <= max(start[1], end[1]):
            return (float(start[0]), float(coord))
    return None

def update_crossing_state(previous_stable, current, line_start, line_end,
                          line_coord, line_direction, margin):
    """Persist a verified side until the track reaches the OTHER verified side.

    Unlike a 1-frame check, this detects gradual passages through the margin band.
    Returns (crossed, direction, crossing_location, new_stable_state).
    """
    side = stable_side(current, line_coord, line_direction, margin)
    if side == 0:
        return False, None, None, previous_stable
    next_state = (side, current)
    if previous_stable is None:
        return False, None, None, next_state
    old_side, old_point = previous_stable
    if old_side == side:
        return False, None, None, next_state
    point = crossing_point(old_point, current, line_start, line_end, line_direction)
    if point is None:
        return False, None, None, next_state
    if line_direction == "horizontal":
        direction = "DOWN" if side > 0 else "UP"
    else:
        direction = "RIGHT" if side > 0 else "LEFT"
    return True, direction, point, next_state

class SpatialDedup:
    """Conservative class/direction-aware spatial-temporal duplicate heuristic.

    It cannot reliably resolve all ID switches. Check dense traffic manually.
    """
    def __init__(self, radius, frame_window, class_rules=None):
        self.radius = radius
        self.frame_window = frame_window
        self.class_rules = class_rules or {}
        self.history_window = max([frame_window] + [v[1] for v in self.class_rules.values()])
        self.records = []

    def is_duplicate(self, point, frame_no, label, direction, track_id):
        radius, window = self.class_rules.get(label, (self.radius, self.frame_window))
        if radius <= 0 or window <= 0:
            return False
        x, y = point
        for fr, rx, ry, cls, direct, old_id in self.records:
            if 0 <= frame_no - fr <= window and cls == label and direct == direction and old_id != track_id:
                if (x-rx)**2 + (y-ry)**2 < radius**2:
                    return True
        return False

    def add(self, point, frame_no, label, direction, track_id):
        x, y = point
        self.records.append((frame_no, x, y, label, direction, track_id))

    def prune(self, current_frame):
        self.records = [r for r in self.records if current_frame-r[0] <= self.history_window]

def draw_count_panel(frame, counts, directions, W, H):
    panel_w, panel_h = 280, 138
    x0 = max(0, W-panel_w-8)
    y0 = 8
    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (min(W-1,x0+panel_w), y0+panel_h), (20,20,20), -1)
    cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)
    cv2.putText(frame, f"Total: {sum(counts.values())}", (x0+8,y0+24),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,255,255), 2)
    row = y0+47
    for cls_id, name in CLASS_NAMES.items():
        d1, d2 = directions
        c1 = counts.get((name,d1),0); c2 = counts.get((name,d2),0)
        cv2.putText(frame, f"{name:10s} {c1+c2:3d} ({d1}:{c1} {d2}:{c2})",
                    (x0+8,row), cv2.FONT_HERSHEY_SIMPLEX, 0.39,
                    CLASS_COLORS.get(cls_id, (255,255,255)), 1)
        row += 21

def run_crossing_logic_tests():
    line_a, line_b, y = (0,100), (300,100), 100
    state = None
    count = 0
    for p in [(100,82), (100,90), (100,96), (100,102), (100,110)]:
        ok, d, hit, state = update_crossing_state(state, p, line_a, line_b, y, "horizontal", 8)
        if ok:
            count += 1
            assert d == "DOWN" and hit == (100.0,100.0)
    assert count == 1, f"Slow crossing should count once: {count}"
    state = None
    count = 0
    for p in [(100,82), (100,95), (100,102), (100,96), (100,106)]:
        ok, _, _, state = update_crossing_state(state, p, line_a, line_b, y, "horizontal", 8)
        count += int(ok)
    assert count == 0, "Within-band jitter must not count"
    state = None
    for p in [(350,80),(350,112)]:
        ok, _, _, state = update_crossing_state(state, p, line_a, line_b, y, "horizontal", 8)
    assert not ok, "Crossing outside finite line must not count"
    test_dedup = SpatialDedup(16,8)
    test_dedup.add((100,100), 10, "motorcycle", "DOWN", 2)
    assert test_dedup.is_duplicate((105,100), 12, "motorcycle", "DOWN", 3)
    assert not test_dedup.is_duplicate((135,100), 12, "motorcycle", "DOWN", 4)
    assert not test_dedup.is_duplicate((105,100), 12, "car", "DOWN", 5)
    assert not test_dedup.is_duplicate((105,100), 12, "motorcycle", "UP", 6)
    print("Passed 5 deterministic counting-logic checks")

run_crossing_logic_tests()

state = None
hits = []
for y in [520, 523, 527, 530, 535, 540, 545, 549, 553, 560]:
    yes, direction, hit, state = update_crossing_state(
        state, (300,y), (70,540), (1200,540), 540, 'horizontal', 3
    )
    if yes:
        hits.append((direction, hit))
assert len(hits) == 1 and hits[0][0] == 'DOWN', hits
print('V5 slow-crossing regression check passed')

review_dedup = SpatialDedup(0, 0, {
    "car": (0, 0), "motorcycle": (0, 0), "bus": (45, 15), "truck": (45, 15)
})
review_dedup.add((663.3, 540), 622, "truck", "DOWN", 1354)
assert review_dedup.is_duplicate((689.0, 540), 633, "truck", "DOWN", 112)
assert not review_dedup.is_duplicate((689.0, 540), 633, "car", "DOWN", 112)
assert not review_dedup.is_duplicate((689.0, 540), 633, "motorcycle", "DOWN", 112)
review_dedup.prune(633)
assert review_dedup.is_duplicate((689.0, 540), 633, "truck", "DOWN", 112)
print('V5.1 targeted truck dedup regression checks passed')


In [ ]:
tracking_model = YOLO(MODEL_NAME)

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise RuntimeError("Cannot re-open video. Upload it again in Step 3.")

raw_output = "/content/vehicle_count_raw.mp4"
writer = cv2.VideoWriter(
    raw_output,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (W, H),
)
if not writer.isOpened():
    cap.release()
    raise RuntimeError("VideoWriter could not create MP4 output.")

counts      = Counter()
counted_ids = set()
track_hist  = {}
track_age   = Counter()
track_stable = {}
suppressed_ids = set()
dedup       = SpatialDedup(DEDUP_RADIUS, DEDUP_FRAMES, DEDUP_CLASS_RULES)
events      = []
processed   = 0
seen_boxes  = 0

diagnostic = Counter()
unique_ids = set()
near_ids = set()
example_crosses = []

directions  = ("DOWN", "UP") if LINE_DIRECTION == "horizontal" else ("RIGHT", "LEFT")
max_frames  = None if MAX_SECONDS is None else max(1, int(MAX_SECONDS * fps))

try:
    while True:
        if max_frames is not None and processed >= max_frames:
            break
        ok, frame = cap.read()
        if not ok:
            break

        frame = cv2.resize(frame, (W, H))
        dedup.prune(processed)

        result = tracking_model.track(
            source=frame,
            persist=True,
            tracker=TRACKER,
            classes=VEHICLE_CLASSES,
            conf=CONFIDENCE,
            iou=IOU_THRESHOLD,
            imgsz=IMAGE_SIZE,
            device=DEVICE,
            verbose=False,
        )[0]

        boxes = result.boxes
        if boxes is not None and boxes.id is not None and len(boxes) > 0:
            coords    = boxes.xyxy.cpu().numpy()
            class_ids = boxes.cls.int().cpu().tolist()
            track_ids = boxes.id.int().cpu().tolist()
            scores    = boxes.conf.cpu().tolist()
            seen_boxes += len(track_ids)

            for xyxy, cls, tid, score in zip(coords, class_ids, track_ids, scores):
                x1, y1, x2, y2 = map(int, xyxy)
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2
                label = CLASS_NAMES.get(cls, f"cls{cls}")
                color = CLASS_COLORS.get(cls, (200, 200, 200))

                unique_ids.add(tid)
                diagnostic['tracked_boxes'] += 1

                perpendicular = cy if LINE_DIRECTION == 'horizontal' else cx
                along = cx if LINE_DIRECTION == 'horizontal' else cy
                lmin = min(line_start[0], line_end[0]) if LINE_DIRECTION == 'horizontal' else min(line_start[1], line_end[1])
                lmax = max(line_start[0], line_end[0]) if LINE_DIRECTION == 'horizontal' else max(line_start[1], line_end[1])
                if abs(perpendicular - line_coordinate) <= 25 and lmin <= along <= lmax:
                    near_ids.add(tid)
                    diagnostic['near_line_box_frames'] += 1

                if tid not in track_hist:
                    track_hist[tid] = deque(maxlen=TRAJECTORY_LEN)
                track_hist[tid].append((cx, cy))
                track_age[tid] += 1

                was_counted = tid in counted_ids or tid in suppressed_ids
                crossed, cross_dir, hit, track_stable[tid] = update_crossing_state(
                    track_stable.get(tid), (cx, cy), line_start, line_end,
                    line_coordinate, LINE_DIRECTION, HYSTERESIS_MARGIN,
                )
                if crossed:
                    diagnostic['geometric_crossings'] += 1
                    if len(example_crosses) < 15:
                        example_crosses.append((processed, tid, label, cross_dir, tuple(round(v,1) for v in hit), track_age[tid]))
                    if was_counted:
                        diagnostic['rejected_already_counted'] += 1
                    elif track_age[tid] < MIN_TRACK_FRAMES:
                        diagnostic['rejected_track_too_short'] += 1
                    elif not inside_roi(*hit, roi_polygon):
                        diagnostic['rejected_outside_roi'] += 1
                    elif ENABLE_SPATIAL_DEDUP and dedup.is_duplicate(hit, processed, label, cross_dir, tid):
                        suppressed_ids.add(tid)
                        diagnostic['rejected_spatial_dedup'] += 1
                    else:
                        counts[(label, cross_dir)] += 1
                        counted_ids.add(tid)
                        dedup.add(hit, processed, label, cross_dir, tid)
                        diagnostic['accepted_crossings'] += 1
                        events.append({
                            "frame": processed,
                            "second": round(processed/fps, 2),
                            "track_id": tid,
                            "vehicle": label,
                            "direction": cross_dir,
                            "confidence": round(score, 3),
                            "cx": round(hit[0], 1),
                            "cy": round(hit[1], 1),
                        })

                box_color  = (0, 220, 0) if tid in counted_ids else color
                thickness  = 3 if tid in counted_ids else 2
                cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, thickness)

                hist = track_hist[tid]
                if len(hist) >= 2:
                    p0 = hist[-2]; p1 = hist[-1]
                    cv2.arrowedLine(frame, p0, p1, box_color, 2, tipLength=0.4)

                tag = f"{label} #{tid} {score:.2f}"
                if tid in counted_ids:
                    tag += " ✓"
                cv2.putText(frame, tag,
                            (max(0, x1), max(16, y1 - 6)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, box_color, 2)

        if roi_polygon is not None:
            cv2.polylines(frame, [roi_polygon], True, (0, 200, 0), 2)
        cv2.line(frame, line_start, line_end, (0, 0, 255), 3)

        if LINE_DIRECTION == "horizontal":
            cv2.line(frame,
                     (line_start[0], line_coordinate - HYSTERESIS_MARGIN),
                     (line_end[0],   line_coordinate - HYSTERESIS_MARGIN),
                     (200, 100, 0), 1)
            cv2.line(frame,
                     (line_start[0], line_coordinate + HYSTERESIS_MARGIN),
                     (line_end[0],   line_coordinate + HYSTERESIS_MARGIN),
                     (200, 100, 0), 1)
        else:
            cv2.line(frame,
                     (line_coordinate - HYSTERESIS_MARGIN, line_start[1]),
                     (line_coordinate - HYSTERESIS_MARGIN, line_end[1]),
                     (200, 100, 0), 1)
            cv2.line(frame,
                     (line_coordinate + HYSTERESIS_MARGIN, line_start[1]),
                     (line_coordinate + HYSTERESIS_MARGIN, line_end[1]),
                     (200, 100, 0), 1)

        draw_count_panel(frame, counts, directions, W, H)

        ts = f"Frame {processed}  {processed/fps:.1f}s"
        cv2.putText(frame, ts, (8, H - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)

        writer.write(frame)
        processed += 1

        if processed % max(1, int(fps * 3)) == 0:
            print(f"  Frame {processed:4d}  {processed/fps:5.1f}s  "
                  f"crossings={sum(counts.values())}  "
                  f"counts={dict(counts)}")

finally:
    cap.release()
    writer.release()

if processed == 0:
    raise RuntimeError("No frames processed. Check the uploaded video.")

print("\n" + "=" * 55)
print(f"Frames processed : {processed}")
print(f"Duration analyzed: {processed/fps:.2f} s")
print(f"Tracked boxes    : {seen_boxes}")
print(f"Total crossings  : {sum(counts.values())}")
print(f"Breakdown        : {dict(counts)}")
print(f"Events logged    : {len(events)}")
print("=" * 55)
if seen_boxes == 0:
    print("WARNING: No tracked vehicles detected. "
          "Try lowering CONFIDENCE or check the video angle.")
if sum(counts.values()) == 0:
    print("NOTE: No crossings counted. "
          "Verify that the counting line crosses the actual vehicle path in the video.")

print("\n=== DIAGNOSTICS ===")
print("Unique tracking IDs:", len(unique_ids))
print("IDs observed near counting line:", len(near_ids))
for k in ['tracked_boxes', 'near_line_box_frames', 'geometric_crossings',
          'accepted_crossings', 'rejected_already_counted',
          'rejected_track_too_short', 'rejected_outside_roi', 'rejected_spatial_dedup']:
    print(f"{k}: {diagnostic[k]}")
print("Example crossing candidates:", example_crosses[:10])
if not near_ids:
    print("ACTION: no tracked centres approach the line. Check line position, video, detections, and tracker stability.")
elif diagnostic['geometric_crossings'] == 0:
    print("ACTION: tracked centres approached the line but never switched confirmed sides with a stable ID. Try HYSTERESIS_MARGIN=0, LINE_RATIO closer to vehicle path, or improve tracking/line position.")
elif not events:
    print("ACTION: geometric crossings occurred but filters rejected them. Inspect the rejection counters above.")
else:
    print("ACTION: crossings are being recorded. Validate their accuracy by manual counting on the same clip.")


In [ ]:
import base64

final_output = "/content/vehicle_count_result.mp4"
if shutil.which("ffmpeg"):
    cmd = [
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", raw_output,
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "23",
        "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        final_output,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("H.264 conversion failed:", result.stderr[-600:])
        final_output = raw_output
else:
    print("ffmpeg unavailable — using raw mp4v file.")
    final_output = raw_output

out_path = Path(final_output)
if not out_path.exists() or out_path.stat().st_size == 0:
    raise FileNotFoundError(f"Output video missing or empty: {final_output}")

print("Output video:", final_output)
print("Size (MB):", round(out_path.stat().st_size / (1024 * 1024), 2))

if final_output != raw_output and out_path.stat().st_size < 40 * 1024 * 1024:
    encoded = base64.b64encode(out_path.read_bytes()).decode("ascii")
    display(HTML(
        '<video width="860" controls playsinline>'
        '<source src="data:video/mp4;base64,' + encoded + '" type="video/mp4">'
        'Your browser does not support HTML5 video.</video>'
    ))
else:
    print("File too large to embed. Download it in Step 12.")


In [ ]:
summary_path = "/content/vehicle_count_summary.csv"
events_path  = "/content/crossing_events.csv"

d1, d2 = directions

with open(summary_path, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["vehicle_type", d1, d2, "total"])
    for cls_id, name in CLASS_NAMES.items():
        c1 = counts.get((name, d1), 0)
        c2 = counts.get((name, d2), 0)
        w.writerow([name, c1, c2, c1 + c2])
    total_d1 = sum(counts.get((n, d1), 0) for n in CLASS_NAMES.values())
    total_d2 = sum(counts.get((n, d2), 0) for n in CLASS_NAMES.values())
    w.writerow(["ALL", total_d1, total_d2, total_d1 + total_d2])

fields = ["frame", "second", "track_id", "vehicle", "direction", "confidence", "cx", "cy"]
with open(events_path, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(events)

print("Saved:", summary_path)
print("Saved:", events_path)
print(f"Total AI count: {sum(counts.values())}")

print("\nSummary:")
print(f"{'Vehicle':<14} {d1:>6} {d2:>6} {'Total':>6}")
print("-" * 36)
for cls_id, name in CLASS_NAMES.items():
    c1 = counts.get((name, d1), 0)
    c2 = counts.get((name, d2), 0)
    print(f"{name:<14} {c1:>6} {c2:>6} {c1+c2:>6}")
print("-" * 36)
print(f"{'ALL':<14} {total_d1:>6} {total_d2:>6} {total_d1+total_d2:>6}")

diag_path = '/content/vehicle_count_diagnostics.csv'
with open(diag_path, 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(['metric', 'value'])
    w.writerow(['processed_frames', processed])
    w.writerow(['unique_tracking_ids', len(unique_ids)])
    w.writerow(['ids_near_line', len(near_ids)])
    for key in ['tracked_boxes','near_line_box_frames','geometric_crossings',
                'accepted_crossings','rejected_already_counted',
                'rejected_track_too_short','rejected_outside_roi','rejected_spatial_dedup']:
        w.writerow([key, diagnostic[key]])
print('Diagnostics CSV:', diag_path)


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

zip_path = "/content/traffic_count_results_v5_1.zip"
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as z:
    for fp in [final_output, summary_path, events_path, diag_path]:
        z.write(fp, arcname=Path(fp).name)
print("Downloading:", zip_path)
files.download(zip_path)


In [ ]:
GROUND_TRUTH = {
    "car":        None,
    "motorcycle": None,
    "bus":        None,
    "truck":      None,
}

ai_by_class = {}
for cls_id, name in CLASS_NAMES.items():
    d1, d2 = directions
    ai_by_class[name] = counts.get((name, d1), 0) + counts.get((name, d2), 0)

ai_total = sum(ai_by_class.values())

if all(v is None for v in GROUND_TRUTH.values()):
    print("=" * 55)
    print("EVALUATION: Ground truth not yet provided.")
    print("=" * 55)
    print()
    print("AI predictions (unverified):")
    print(f"{'Class':<14} {'AI':>6}")
    print("-" * 22)
    for name, cnt in ai_by_class.items():
        print(f"{name:<14} {cnt:>6}")
    print("-" * 22)
    print(f"{'TOTAL':<14} {ai_total:>6}")
    print()
    print("HOW TO COUNT MANUALLY:")
    print("1. Download the annotated video from Step 12.")
    print("2. Open it in VLC or any video player.")
    print("3. Watch at reduced speed (0.5x or 0.25x).")
    print("4. Count each vehicle that crosses the RED LINE.")
    print("5. Record counts per vehicle type (car / motorcycle / bus / truck).")
    print("6. Enter values in GROUND_TRUTH above and re-run this cell.")
else:
    print("=" * 60)
    print("EVALUATION RESULTS")
    print("=" * 60)
    print(f"{'Class':<14} {'GT':>6} {'AI':>6} {'AbsErr':>8} {'RelErr%':>9}")
    print("-" * 50)

    total_gt = 0
    total_ai_known = 0
    abs_errors = []
    complete_gt = all(v is not None for v in GROUND_TRUTH.values())
    for name in CLASS_NAMES.values():
        gt  = GROUND_TRUTH.get(name)
        ai  = ai_by_class.get(name, 0)
        if gt is None:
            print(f"{name:<14} {'N/A':>6} {ai:>6}")
            continue
        err = abs(ai - gt)
        rel = f"{100 * err / gt:.1f}" if gt > 0 else "N/A"
        abs_errors.append(err)
        total_gt += gt
        total_ai_known += ai
        print(f"{name:<14} {gt:>6} {ai:>6} {err:>8} {rel:>9}")
    print("-" * 50)
    total_err = abs(total_ai_known - total_gt)
    total_rel = f"{100 * total_err / total_gt:.1f}" if total_gt > 0 else "N/A"
    if complete_gt:
        print(f"{'TOTAL':<14} {total_gt:>6} {ai_total:>6} {total_err:>8} {total_rel:>9}")
    else:
        print("TOTAL comparison deferred: manual counts missing for one or more classes")
    print()
    print("Event-level metrics:")
    print("  To compute event-level Precision/Recall/F1, compare crossing_events.csv")
    print("  against a manually-annotated event list (frame number + vehicle type).")
    print("  Match predicted events to ground-truth events using a time tolerance")
    print("  (e.g., ±15 frames) and class label.")
    print()
    if abs_errors:
        mae = sum(abs_errors) / len(abs_errors)
        print(f"  Mean Absolute Error (per class): {mae:.2f}")
    print()
    print("Diagnosis:")
    if complete_gt and total_gt > 0:
        print(f"  Relative total count error: {100 * total_err/total_gt:.1f}%")
        print("  Compare false positives and missed vehicles by replaying crossing events.")
    else:
        print("  Enter all class counts to evaluate the total count error.")


In [ ]:
comparison_table = [

    ("A", "yolov8n", "bytetrack", 960,  None, None, None, "baseline"),
    ("B", "yolov8s", "bytetrack", 960,  None, None, None, ""),
    ("C", "yolov8n", "botsort",   960,  None, None, None, ""),
    ("D", "yolov8n", "bytetrack", 1280, None, None, None, ""),
]

print("Comparison table (fill in AI_Total and AbsErr after each run):")
print(f"{'Cfg':<4} {'Model':<10} {'Tracker':<12} {'imgsz':<6} "
      f"{'AI':>5} {'GT':>5} {'Err':>5} {'Notes'}")
print("-" * 65)
for row in comparison_table:
    cfg, model, tracker, sz, ai, gt, err, notes = row
    ai_s  = str(ai)  if ai  is not None else "---"
    gt_s  = str(gt)  if gt  is not None else "---"
    err_s = str(err) if err is not None else "---"
    print(f"{cfg:<4} {model:<10} {tracker:<12} {sz:<6} "
          f"{ai_s:>5} {gt_s:>5} {err_s:>5} {notes}")
print()
print("Interpretation guidance:")
print("  yolov8n vs yolov8s : s is ~2–4% more accurate on COCO but ~2x slower on CPU.")
print("  bytetrack vs botsort: botsort uses ReID features; better ID stability but slower.")
print("  imgsz 960 vs 1280  : 1280 improves detection of small/distant vehicles.")
print("  CONFIDENCE 0.20 vs 0.25 vs 0.30: trade-off between recall and precision.")
